<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.lab.1. Wisconsin Cancer Classification - Neural Networks
  </strong>
</div>

- [ ] Implementación básica de SVM con kernel lineal y rBF
- [ ] Comparación de rendimiento entre los dos modelos
- [ ] Una visualización mostrando el límite de decisión


In [40]:
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from rich.table import Table
from rich.console import Console
from sklearn import svm

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

Visual configuration of Seaborn for more attractive graphics

In [41]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

Rich console instance to display results

In [42]:
console = Console()

## Data Loading

In [43]:
breast_cancer = load_breast_cancer(as_frame=True)
X = breast_cancer.data
y = breast_cancer.target

target_names = breast_cancer.target_names

diagnosis_labels = breast_cancer.target.map({0: target_names[0], 1: target_names[1]})

## Preprocessing of data

Split the data with all features into training and testing sets

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (455, 30)
Test set size: (114, 30)


Method to escalate the data

In [45]:
def scaler_data(scaler, X_train, X_test):
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

## Training Methods

Method to train a regression model

In [46]:
def train_regression_model(X_train_scaled, y_train_data, X_test_scaled):

    model = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000)
    model.fit(X_train_scaled, y_train_data)

    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    return y_train_pred, y_test_pred

Method to train a SVM Linear

In [60]:
def train_svm_model(kernel, X_train_scaled, y_train_data, X_test_scaled):
    svm_model = svm.SVC(kernel=kernel, C=10000)
    svm_model.fit(X_train_scaled, y_train_data)

    svm_y_train_pred = svm_model.predict(X_train_scaled)
    svm_y_test_pred = svm_model.predict(X_test_scaled)

    return svm_y_train_pred, svm_y_test_pred

## Evaluation Methods

Method to calculate metrics

In [48]:
def calculate_metrics_for_set(y_test, y_test_pred):
    acc = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)

    return acc, precision, recall, f1

def calculate_metrics(
    y_train_regression_model, y_train_regression_model_pred, y_test_model, y_test_model_pred
):

    acc_train, precision_train, recall_train, f1_train = calculate_metrics_for_set(
        y_train_regression_model, y_train_regression_model_pred
    )
    acc_test, precision_test, recall_test, f1_test = calculate_metrics_for_set(
        y_test_model, y_test_model_pred
    )

    return {
        "accuracy_score_train": acc_train,
        "precision_score_train": precision_train,
        "recall_score_train": recall_train,
        "f1_score_train": f1_train,
        "accuracy_score_test": acc_test,
        "precision_score_test": precision_test,
        "recall_score_test": recall_test,
        "f1_score_test": f1_test,
    }

In [49]:
def to_tensors(X_train, y_train, X_test, y_test):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

    return X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor

## Visualization methods

Method to create the comparison table

In [50]:
def create_comparison_table(title, results):

    comparison_table_normalization = Table(title=title)

    comparison_table_normalization.add_column(
        "Model", justify="center", style="green", width=15
    )
    comparison_table_normalization.add_column(
        "Accuracy-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Accuracy-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Precision-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Precision-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Recall-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Recall-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "F1-Train", justify="right", style="green"
    )
    comparison_table_normalization.add_column("F1-Test", justify="right", style="green")

    for model_name, metrics in results.items():
        comparison_table_normalization.add_row(
            model_name,
            f'{metrics["accuracy_score_train"]:.4f}',
            f'{metrics["accuracy_score_test"]:.4f}',
            f'{metrics["precision_score_train"]:.4f}',
            f'{metrics["precision_score_test"]:.4f}',
            f'{metrics["recall_score_train"]:.4f}',
            f'{metrics["recall_score_test"]:.4f}',
            f'{metrics["f1_score_train"]:.4f}',
            f'{metrics["f1_score_test"]:.4f}',
        )

    console.print(comparison_table_normalization)

Method to show the confusion matrix

In [51]:
def plot_confusion_matrix(cm, title, ax):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")


def plot_confusion_matrix_comparison(y_test, y_test_pred_log):
    cm_log = confusion_matrix(y_test, y_test_pred_log)
    #cm_nn = confusion_matrix(y_test, y_test_pred_nn)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    plot_confusion_matrix(cm_log, "Confusion Matrix - Logistic Regression", axes[0])
    #plot_confusion_matrix(cm_nn, "Confusion Matrix - Neural Network (128,128)", axes[1])

    plt.tight_layout()
    plt.show()

Where the metrics information of each model will be storaged

In [53]:
comparison_metrics = {}

The data will be normalized using a standard scaler.

In [75]:
scaler = StandardScaler()
X_train_scaled, X_test_scaled = scaler_data(scaler, X_train, X_test)

## 1. Regression Model

In [76]:
rg_y_train_pred_log, rg_y_test_pred_log = train_regression_model(X_train_scaled, y_train, X_test_scaled)
metrics = calculate_metrics(y_train, rg_y_train_pred_log, y_test, rg_y_test_pred_log)
comparison_metrics["Logistic Regression"] = metrics

## 2 SVM Linear

In [77]:
svm_y_train_pred_log, svm_y_test_pred_log = train_svm_model("linear", X_train_scaled, y_train, X_test_scaled)
metrics = calculate_metrics(y_train, svm_y_train_pred_log, y_test, svm_y_test_pred_log)
comparison_metrics["SVM Linear"] = metrics

In [78]:
create_comparison_table(
    "Comparison of metrics between Neural Network and Logistic Regression with standard normalization",
    comparison_metrics,
)

       Comparison of metrics between Neural Network and Logistic Regression with standard normalization       
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃                 ┃ Accuracy- ┃ Accuracy- ┃ Precision- ┃ Precision- ┃ Recall- ┃ Recall- ┃          ┃         ┃
┃      Model      ┃     Train ┃      Test ┃      Train ┃       Test ┃   Train ┃    Test ┃ F1-Train ┃ F1-Test ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│    Logistic     │    0.9868 │    0.9737 │     0.9861 │     0.9722 │  0.9930 │  0.9859 │   0.9895 │  0.9790 │
│   Regression    │           │           │            │            │         │         │          │         │
│   SVM Linear    │    1.0000 │    0.9386 │     1.0000 │     0.9848 │  1.0000 │  0.9155 │   1.0000 │  0.9489 │
└─────────────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴──────────┴─────────┘